In [1]:
import time
import os
import numpy as np
from pathlib import Path
import random

from concurrent.futures import ProcessPoolExecutor as Exe
from metasmith.coms.ipc import RemoteShell, ConnectionError
from local.constants import WORKSPACE_ROOT

In [2]:
# import pty
# import os
# import time

# mypid = os.getpid()
# print("pid", mypid)
# from metasmith.coms.ipc import TerminalProcess, NonBlockingReader, RemoteShell, PipeServer, PipeClient

# N=1000
# for i in range(N):
#     # with TerminalProcess() as shell:
#     #     shell.Write("echo x")

#     # out_master, out_slave = pty.openpty()
#     # err_master, err_slave = pty.openpty()
#     # _fds = [out_master, err_master]
#     # with NonBlockingReader(out_master):
#     #     pass
#     # for fd in _fds:
#     #     os.close(fd)

#     # with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in") as shell:
#     #     res = shell.Exec("echo asdf", history=True, timeout=1)
#     #     assert "asdf" in res.out

#     # print(i, end="")
#     with PipeServer(Path("./cache"), lambda x, s: None, id=f"{i}") as con:
#         # time.sleep(0.1)
#         # print("s", end="")
#         with PipeClient(Path(f"./cache/{i}.in"), timeout=1) as client:
#             time.sleep(0.1)
#     #         print("c", end="")
#     #     print("C", end="")
#     # print("S")
#     if i%(N//10)==0: os.system(f"ls /proc/{mypid}/fd | wc")

In [3]:
# # import os
# # import select

# a, b = os.pipe()
# # x, y = os.pipe()

# os.write(b, b"123")
# # r, _, _ = select.select([a], [], [], 3)

# x = os.read(a, 16)
# x = int(x)
# x

In [4]:
with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=3) as shell:
    res = shell.Exec("sleep 5 && echo asdf", history=True)
res

AttributeError: 'PipeClient' object has no attribute '_keep_alive_worker'

In [4]:
def r(args):
    i = args
    for retry in range(5):
        try:
            with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=60) as shell:
                res = shell.Exec("echo asdf", history=True, timeout=3)
                assert "asdf" in res.out
                return i
        except (TimeoutError, ConnectionError):
            a, b = 2**(retry-1), 2**retry
            dt = random.random()*(b-a) + a
            print(f"[{i}] timeout, retry in [{dt}]s")
            time.sleep(dt)

mypid = os.getpid()
fd_path = Path(f"/proc/{mypid}/fd")
n_fd = len(list(fd_path.iterdir()))
print(f"start: {n_fd}")
k = 1000
with Exe(max_workers=100) as exe:
    for res in exe.map(r, list(range(k))):
        if res is None: continue
        n_fd = len(list(fd_path.iterdir()))
        print(f"{res} {n_fd}", end="\r")

# for i in range(k):
#     r(i)
#     print(i, end="\r")

start: 74
[97] timeout, retry in [0.7914486805016839]s
[133] timeout, retry in [0.8210336761859343]s
[4] timeout, retry in [0.8662272818875962]s
[55] timeout, retry in [0.8406954788242741]s
[110] timeout, retry in [0.6857081769027285]s
[27] timeout, retry in [0.6077837931954956]s
[117] timeout, retry in [0.9223966024630073]s
[35] timeout, retry in [0.6864594045913217]s
[12] timeout, retry in [0.9778035864627002]s
[51] timeout, retry in [0.9266897405459689]s
[80] timeout, retry in [0.9628392972021058]s
[120] timeout, retry in [0.6749958627873826]s
[51] timeout, retry in [1.1389540835570786]s
[181] timeout, retry in [0.8724326102753912]s
[63] timeout, retry in [0.8370365934705174]s
[209] timeout, retry in [0.8004044952221607]s
[215] timeout, retry in [0.8934420552449722]s
[184] timeout, retry in [0.9878606957941761]s
[195] timeout, retry in [0.7442248488594724]s
[213] timeout, retry in [0.7391552043437738]s
[122] timeout, retry in [0.5531856288445187]s
[195] timeout, retry in [1.76252651

KeyboardInterrupt: 

In [ ]:
# import pty
# import os
# import time

# mypid = os.getpid()
# print("pid", mypid)

# pids = []
# for i in range(200):
#     a, b = pty.openpty()
#     pids += [a, b]
# os.system(f"ls /proc/{mypid}/fd | wc")
# for p in pids:
#     os.close(p)
# os.system(f"ls /proc/{mypid}/fd | wc")